# Local MCP 를 이용한 Responses API 호출

- 로컬에서 실행 중인 FastMCP 서버(clothing_price_server)에 OpenAI 모델(gpt-5-mini)이 연결되어, 서버의 도구(list_items 등)를 사용함.  
- ngrok을 통해 외부에서 접근 가능한 HTTPS URL을 부여함.

In [15]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-5-nano"

In [17]:
ngrok_url = "https://lurchingly-unenticed-yuette.ngrok-free.dev"

print(f"{ngrok_url}/mcp")

https://lurchingly-unenticed-yuette.ngrok-free.dev/mcp


In [18]:
# -------------------------------------------------------------
# MCP 서버 연결 정보 설정
# -------------------------------------------------------------
# tools 파라미터 안에 'type': 'mcp' 를 지정하면,
# 모델이 외부 MCP 서버에 정의된 도구(tool)들을 불러와 사용할 수 있습니다.
#
# - server_label: MCP 서버를 식별하기 위한 이름 (자유롭게 지정 가능)
# - server_url:   MCP 서버의 HTTP 또는 HTTPS 엔드포인트 URL
#
# ※ 주의
#   - localhost 주소는 클라우드(OpenAI API)에서 접근할 수 없습니다.
#   - 따라서 ngrok 또는 cloudflared 같은 터널링 서비스를 사용해야 합니다.
#   - ngrok 실행 예시:
#       ngrok http 3000
#     실행 후 나온 URL(예: https://xxxx.ngrok-free.dev)을 아래에 입력.
# -------------------------------------------------------------

response = client.responses.create(
    model=Model,  # 사용할 모델 지정
    tools=[
        {
            "type":         "mcp",                          # MCP 서버 타입 지정
            "server_label": "clothing_price_server",        # 서버 식별용 라벨
            # "server_url": "http://localhost:3000/mcp",    # (로컬 환경 전용 - 클라우드 접근 불가)
            "server_url":   f"{ngrok_url}/mcp",  # ngrok 공개 URL
        }
    ],
    input="현재 등록된 모든 의류 목록을 보여줘."  # MCP 서버의 'list_items' 도구를 사용할 수 있음
)

### OpenAI Responses API - MCP 승인(approval) 응답 처리
Responses API에서 MCP 서버 호출 시, 보안상 사용자의 승인이 필요한 도구(tool)가 있을 경우 모델이 "mcp_approval_request" 형태의 응답을 생성합니다.

- response.output 중에서 type이 "mcp_approval_request"인 항목을 찾음
- 찾은 approval 객체의 ID를 변수로 저장
- 이후 이 ID를 이용해 승인(approve) 또는 거부(reject) 응답을 보낼 수 있음
- 모델이 단순히 정보 조회(read-only) 도구를 사용할 때는 자동 실행되지만, 상태를 바꾸거나 외부와 상호작용(write, send, delete 등) 하는 도구일 경우 사용자의 “승인” 요청

In [19]:
# 응답 출력 구조 확인
# - McpListTools: MCP 서버에서 사용 가능한 도구 목록
# - ResponseReasoningItem: 모델의 추론 과정
# - McpApprovalRequest: 도구 실행 전 사용자 승인 요청
response.output

[McpListTools(id='mcpl_0c1d225397ae9dc00069b62b50e484819685fdc71b2142dad8', server_label='clothing_price_server', tools=[McpListToolsTool(input_schema={'additionalProperties': False, 'properties': {'item': {'type': 'string'}}, 'required': ['item'], 'type': 'object'}, name='get_price', annotations={'read_only': False}, description='의류 품목의 가격을 조회합니다. 항상 (found, price)를 반환합니다.'), McpListToolsTool(input_schema={'additionalProperties': False, 'properties': {'item': {'type': 'string'}, 'price': {'type': 'number'}}, 'required': ['item', 'price'], 'type': 'object'}, name='add_item', annotations={'read_only': False}, description='의류 품목을 추가하거나 가격을 업데이트합니다. 항상 (item, price)를 반환합니다.'), McpListToolsTool(input_schema={'additionalProperties': False, 'properties': {}, 'type': 'object'}, name='list_items', annotations={'read_only': False}, description='모든 의류 품목과 가격 목록을 반환합니다.')], type='mcp_list_tools', error=None),
 ResponseReasoningItem(id='rs_0c1d225397ae9dc00069b62b52a3888196b3a39aed5af9611a', summa

In [20]:
# 승인 요청(mcp_approval_request) 항목만 필터링하는 제너레이터
(o for o in response.output if o.type == "mcp_approval_request")

<generator object <genexpr> at 0x00000283BD54E190>

In [21]:
# 승인 요청 객체를 꺼내고, 승인 ID를 저장
# 이 ID를 사용하여 다음 단계에서 승인(approve) 응답을 보냄
approval = next(o for o in response.output if o.type == "mcp_approval_request")
approval_id = approval.id
approval_id

'mcpr_0c1d225397ae9dc00069b62b53cdf08196a0be5ce744ec274d'

In [22]:
# 승인 응답을 보내 도구 실행을 허가
# - approval_request_id: 위에서 받은 승인 요청 ID
# - approve: True로 설정하면 도구 실행 승인
# - previous_response_id: 이전 응답과 연결하여 대화 흐름 유지
response2 = client.responses.create(
    model=Model,
    input=[
        {
            "type": "mcp_approval_response",
            "approval_request_id": approval_id,  
            "approve": True,                     
        }
    ],
    tools=[
        {
            "type":         "mcp",
            "server_label": "clothing_price_server",
            "server_url":   f"{ngrok_url}/mcp",
        }
    ],
    previous_response_id = response.id
)

response2.output_text

'다음은 현재 등록된 의류 목록과 가격입니다:\n\n- glove: 5.00\n- hoodie: 39.95\n- jeans: 59.90\n- t-shirt: 19.99\n\n다른 항목을 추가하거나 가격 정보를 업데이트 원하시면 알려주세요.'

### MCP 도구를 이용하여 새로운 품목 등록

In [23]:
# 새로운 품목 등록 요청
# add_item 도구를 호출하므로 승인 요청이 발생함
response = client.responses.create(
    model=Model,
    tools=[
        {
            "type":         "mcp",
            "server_label": "clothing_price_server",
            "server_url":   f"{ngrok_url}/mcp",
        }
    ],
    input="신제품인 glove 를 5 달러로 새로 등록해줘."  
)

# 승인 요청 ID 추출
approval = next(o for o in response.output if o.type == "mcp_approval_request")
approval_id = approval.id
approval_id

'mcpr_0957a7f9684a18000069b62b8bac408195a302d892f3d6b125'

In [24]:
# 품목 등록(add_item) 승인 후 결과 확인
response3 = client.responses.create(
    model="gpt-5-mini",
    input=[
        {
            "type": "mcp_approval_response",
            "approval_request_id": approval_id,  
            "approve": True,                     
        }
    ],
    tools=[
        {
            "type":         "mcp",
            "server_label": "clothing_price_server",
            "server_url":   f"{ngrok_url}/mcp",
        }
    ],
    previous_response_id = response.id
)

response3.output_text

'신제품 "glove"을(를) 가격 5 달러로 등록했습니다. 다른 항목도 추가할까요?'

In [25]:
# 등록 후 전체 목록 다시 조회하여 glove가 추가되었는지 확인
response = client.responses.create(
    model=Model,
    tools=[
        {
            "type":         "mcp",
            "server_label": "clothing_price_server",
            "server_url":   f"{ngrok_url}/mcp",
        }
    ],
    input="현재 등록된 모든 의류 목록을 보여줘."
)

# 승인 요청 ID 추출
approval = next(o for o in response.output if o.type == "mcp_approval_request")
approval_id = approval.id
approval_id

'mcpr_0042a6be8e2628e00069b62b92e78c8194b48992cc9f5f3e81'

In [26]:
# 목록 조회 승인 후 결과 확인 (glove 포함 여부 확인)
response4 = client.responses.create(
    model="gpt-5-mini",
    input=[
        {
            "type": "mcp_approval_response",
            "approval_request_id": approval_id,  
            "approve": True,                     
        }
    ],
    tools=[
        {
            "type":         "mcp",
            "server_label": "clothing_price_server",
            "server_url":   f"{ngrok_url}/mcp",
        }
    ],
    previous_response_id = response.id
)

response4.output_text

'현재 등록된 의류 목록은 다음과 같습니다:\n\n- glove — $5.00  \n- hoodie — $39.95  \n- jeans — $59.90  \n- t-shirt — $19.99\n\n원하시면 항목 추가·수정하거나 상세 조회해 드릴까요?'